In [1]:
pip install pandas scikit-learn nltk rapidfuzz

  Using cached pandas-3.0.1-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached nltk-3.9.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached rapidfuzz-3.14.3-cp311-cp311-win_amd64.whl.metadata (12 kB)
  Using cached numpy-2.4.3-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached regex-2026.2.28-cp311-cp311-win_amd64.whl.metadata (41 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached pandas-3.0.1-cp311-cp311-win_amd64.whl (9.9 MB)
Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl (8.1 MB)
Using cached nltk-3.9.3-py3-none-any.whl (1.5 MB)


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# ===============================
# 1. IMPORT LIBRARIES
# ===============================
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from rapidfuzz import process
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# ===============================
# 2. LOAD DATASET
# ===============================
df = pd.read_csv("simple_disease_dataset_200_rows.csv")

# Separate features and target
X = df.drop(["Disease", "Severity"], axis=1)
y = df["Disease"]

# Save symptom column names
all_symptoms = list(X.columns)

# ===============================
# 3. TRAIN ML MODEL
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier()
model.fit(X_train, y_train)

print("Model Accuracy:", model.score(X_test, y_test))

# ===============================
# 4. SYMPTOM SYNONYM DICTIONARY
# ===============================
symptom_dict = {
    "tired": "fatigue",
    "fatigue": "fatigue",
    "high fever": "high_fever",
    "fever": "fever",
    "cold": "runny_nose",
    "running nose": "runny_nose",
    "sore throat": "sore_throat",
    "head pain": "headache",
    "stomach ache": "stomach_pain",
    "loose motion": "diarrhea",
    "throwing up": "vomiting",
    "breathing problem": "breathlessness"
}

# ===============================
# 5. NLP PROCESS FUNCTION
# ===============================
def extract_symptoms(user_text):

    # Step 1: Lowercase
    text = user_text.lower()

    # Step 2: Tokenization
    words = word_tokenize(text)

    # Step 3: Remove Stopwords
    stop_words = set(stopwords.words("english"))
    filtered_words = [w for w in words if w not in stop_words]

    processed_text = " ".join(filtered_words)

    extracted = []

    # Step 4: Phrase Matching
    for key in symptom_dict:
        if key in processed_text:
            extracted.append(symptom_dict[key])

    # Step 5: Fuzzy Matching (for single words)
    for word in filtered_words:
        match, score, _ = process.extractOne(word, all_symptoms)
        if score > 80:
            extracted.append(match)

    return list(set(extracted))

# ===============================
# 6. CONVERT TO FEATURE VECTOR
# ===============================
def symptoms_to_vector(symptoms):
    return [1 if s in symptoms else 0 for s in all_symptoms]

# ===============================
# 7. PREDICTION FUNCTION
# ===============================
def predict_disease(user_input):

    symptoms = extract_symptoms(user_input)
    vector = symptoms_to_vector(symptoms)

    prediction = model.predict([vector])[0]

    print("Extracted Symptoms:", symptoms)
    print("Predicted Disease:", prediction)

# ===============================
# 8. TEST EXAMPLE
# ===============================
user_text = "I feel tired and have high fever with sore throat"

predict_disease(user_text)

Model Accuracy: 0.775
Extracted Symptoms: ['sore_throat', 'fever', 'high_fever', 'fatigue']
Predicted Disease: Viral Fever


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
c:\Users\ADMIN\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [5]:
import pandas as pd

# Load dataset
df = pd.read_csv("simple_disease_dataset_200_rows.csv")

# Get symptom column names
all_symptoms = list(df.drop(["Disease", "Severity"], axis=1).columns)

def symptoms_to_vector(extracted_symptoms):
    return [1 if symptom in extracted_symptoms else 0 for symptom in all_symptoms]


# Example extracted symptoms
user_symptoms = ['fever', 'sore_throat', 'fatigue']

vector = symptoms_to_vector(user_symptoms)

print("Feature Vector:")
print(vector)

Feature Vector:
[1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [7]:
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_csv("simple_disease_dataset_200_rows.csv")

# Features
X = df.drop(["Disease", "Severity"], axis=1)

# Targets
y_disease = df["Disease"]
y_severity = df["Severity"]

# Save symptom columns
symptom_columns = X.columns.tolist()

# Train-test split
X_train, X_test, y_d_train, y_d_test = train_test_split(
    X, y_disease, test_size=0.2, random_state=42
)

_, _, y_s_train, y_s_test = train_test_split(
    X, y_severity, test_size=0.2, random_state=42
)

# Train models
disease_model = RandomForestClassifier(n_estimators=100)
disease_model.fit(X_train, y_d_train)

severity_model = RandomForestClassifier(n_estimators=100)
severity_model.fit(X_train, y_s_train)

# Evaluate
disease_pred = disease_model.predict(X_test)
severity_pred = severity_model.predict(X_test)

print("Disease Model Accuracy:", accuracy_score(y_d_test, disease_pred))
print("Severity Model Accuracy:", accuracy_score(y_s_test, severity_pred))

# Save models
pickle.dump(disease_model, open("disease_model.pkl", "wb"))
pickle.dump(severity_model, open("severity_model.pkl", "wb"))
pickle.dump(symptom_columns, open("symptom_columns.pkl", "wb"))

print("Models saved successfully!")

Disease Model Accuracy: 0.8
Severity Model Accuracy: 0.875
Models saved successfully!
